# 🧠 Nova v2.7.0 — Fine-Tuning con QLoRA

**Este notebook entrena tu modelo Nova personalizado.**

Datasets:
- `somosnlp/somos-clean-alpaca-es` (52K instrucciones en español)
- `iamtarun/code_instructions_120k_alpaca` (120K instrucciones de código)

**Instrucciones:**
1. Ve a `Entorno de ejecución` → `Cambiar tipo de entorno` → Selecciona **T4 GPU**
2. Haz clic en `Entorno de ejecución` → `Ejecutar todo`
3. Espera ~1-2 horas
4. Descarga el archivo GGUF al final

---

## Paso 1: Instalar Dependencias

In [ ]:
%%capture
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets
print('✅ Dependencias instaladas')

## Paso 2: Cargar Modelo Base

In [ ]:
from unsloth import FastLanguageModel
import torch

# Configuración del modelo
max_seq_length = 2048
dtype = None  # Auto-detect
load_in_4bit = True  # QLoRA 4-bit

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="deepseek-ai/deepseek-coder-1.3b-instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print('✅ Modelo cargado: deepseek-coder-1.3b-instruct')

## Paso 3: Configurar LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                    # Rank de LoRA (16 = buen balance calidad/velocidad)
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,          # Optimizado: 0 es más rápido
    bias="none",
    use_gradient_checkpointing="unsloth",  # Ahorra 30% VRAM
    random_state=3407,
)

print('✅ LoRA configurado (rank=16, 7 capas objetivo)')

## Paso 4: Cargar y Combinar Datasets

In [ ]:
from datasets import load_dataset, concatenate_datasets

# --- Dataset 1: Español General (52K) ---
print('📥 Descargando dataset español...')
ds_spanish = load_dataset("somosnlp/somos-clean-alpaca-es", split="train")
print(f'   → {len(ds_spanish)} ejemplos en español')

# --- Dataset 2: Código Multi-lenguaje (120K) ---
print('📥 Descargando dataset de código...')
ds_code = load_dataset("iamtarun/code_instructions_120k_alpaca", split="train")
print(f'   → {len(ds_code)} ejemplos de código')

# --- Unificar columnas ---
# El dataset español puede tener columnas diferentes, las normalizamos
def normalize_spanish(example):
    return {
        "instruction": example.get("instruction", ""),
        "input": example.get("input", ""),
        "output": example.get("output", ""),
    }

def normalize_code(example):
    return {
        "instruction": example.get("instruction", ""),
        "input": example.get("input", ""),
        "output": example.get("output", ""),
    }

ds_spanish = ds_spanish.map(normalize_spanish, remove_columns=ds_spanish.column_names)
ds_code = ds_code.map(normalize_code, remove_columns=ds_code.column_names)

# --- Combinar ---
dataset = concatenate_datasets([ds_spanish, ds_code])
dataset = dataset.shuffle(seed=42)

print(f'\n✅ Dataset combinado: {len(dataset)} ejemplos totales')

## Paso 5: Formatear para Entrenamiento

In [ ]:
# Template de prompt para Nova
nova_prompt_template = """### Instrucción:
{instruction}

### Entrada:
{input}

### Respuesta:
{output}"""

EOS_TOKEN = tokenizer.eos_token

def format_prompts(examples):
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]
    texts = []
    for inst, inp, out in zip(instructions, inputs, outputs):
        inp_text = inp if inp else "(ninguna)"
        text = nova_prompt_template.format(
            instruction=inst,
            input=inp_text,
            output=out
        ) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(format_prompts, batched=True)

# Mostrar un ejemplo
print('📋 Ejemplo de prompt formateado:')
print('=' * 50)
print(dataset[0]['text'][:500])
print('...')

## Paso 6: Entrenar 🚀

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=50,
        num_train_epochs=1,           # 1 época = suficiente para 172K ejemplos
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=50,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="nova_training_output",
        save_strategy="no",           # No guardar checkpoints intermedios (ahorrar espacio)
    ),
)

print('🚀 Iniciando entrenamiento...')
print(f'   Ejemplos: {len(dataset)}')
print(f'   Épocas: 1')
print(f'   Batch effectivo: {2 * 4} = 8')
print(f'   Pasos estimados: ~{len(dataset) // 8}')
print()

stats = trainer.train()

print(f'\n✅ Entrenamiento completado!')
print(f'   Loss final: {stats.training_loss:.4f}')
print(f'   Tiempo: {stats.metrics["train_runtime"]:.0f} segundos')

## Paso 7: Probar el Modelo

In [ ]:
# Prueba rápida antes de exportar
FastLanguageModel.for_inference(model)

test_prompts = [
    "Hazme un hola mundo en Python",
    "¿Qué es una variable en programación?",
    "Crea una página HTML con un botón que cambie de color",
]

for prompt in test_prompts:
    inputs = tokenizer(
        f"### Instrucción:\n{prompt}\n\n### Entrada:\n(ninguna)\n\n### Respuesta:\n",
        return_tensors="pt"
    ).to("cuda")
    
    outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.5)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    print(f'\n📝 Prompt: {prompt}')
    print(f'💬 Nova: {response.split("### Respuesta:")[-1].strip()[:300]}')
    print('-' * 50)

## Paso 8: Exportar a GGUF (Para Ollama)

In [ ]:
print('📦 Exportando modelo a GGUF (esto tarda ~5-10 min)...')

model.save_pretrained_gguf(
    "nova-finetuned",
    tokenizer,
    quantization_method="q4_k_m"  # Mejor calidad que Q4_0 con tamaño similar
)

print('✅ Modelo exportado: nova-finetuned/unsloth.Q4_K_M.gguf')

## Paso 9: Descargar el Modelo

Ejecuta esta celda y descarga el archivo `.gguf`.
Luego ponlo en la carpeta `Nova/training/` de tu PC.

In [ ]:
from google.colab import files
import os

gguf_path = "nova-finetuned/unsloth.Q4_K_M.gguf"

if os.path.exists(gguf_path):
    size_mb = os.path.getsize(gguf_path) / (1024 * 1024)
    print(f'📂 Archivo: {gguf_path}')
    print(f'📏 Tamaño: {size_mb:.0f} MB')
    print(f'\n⬇️ Descargando... (esto puede tardar unos minutos)')
    files.download(gguf_path)
else:
    print('❌ Error: No se encontró el archivo GGUF.')
    print('   Verifica que el paso anterior se completó correctamente.')

---

## ✅ ¡Listo!

Una vez descargado el archivo `.gguf`:

1. Ponlo en `C:\Users\DELL\Desktop\inventos\Nova\training\`
2. Ejecuta `import_model.bat` (doble clic)
3. ¡Nova ahora habla español y programa en múltiples lenguajes!

---
*Nova v2.7.0 — Fine-tuned con ❤️ por Starlyn*